<a href="https://colab.research.google.com/github/sxjithh/Core-Scripts/blob/master/RAG_EXAMBUDDY_AI_study_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 — Practical RAG: ExamBuddy - AI study Assistant

## Student Notebook — FREE Google Colab

We will build the syllabus project step by step:

**PDFs → Text → Chunks → Metadata → Embeddings → Vector Database → Similarity Search → Semantic Search → RAG → Answer → Citations → Multi-document Search → Evaluation**

No OpenAI API credits are required.

## How to use this notebook

For each stage:

**1. Understand the concept → 2. Run the code → 3. Observe the result → 4. Move to the next stage.**

The application is the practical demonstration of every syllabus topic.

# 1. Install the required libraries

- `pypdf` → PDF text extraction
- `sentence-transformers` → free local embeddings
- `chromadb` → local vector database
- `transformers` → local LLM
- `accelerate` → model runtime

No paid API is needed.

In [1]:
!pip install -q pypdf sentence-transformers chromadb transformers accelerate
print("✅ Installation complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

# 2. Upload study materials PDFs

Upload one or more PDFs, for example:

- Leave and Holiday Policy
- Work From Home Policy
- Employee Handbook
- Travel Policy

Multiple PDFs let us demonstrate **multi-document search**.

In [18]:
from google.colab import files

uploaded_files = files.upload()

print(f"✅ Uploaded {len(uploaded_files)} file(s)")
for filename in uploaded_files:
    print(" -", filename)

Saving 3029.pdf to 3029.pdf
✅ Uploaded 1 file(s)
 - 3029.pdf


# 3. PDF → Text

First we extract text from every PDF page.

We also keep **metadata**:

- `source` = PDF filename
- `page` = page number

This metadata will later be used for citations.

In [19]:
from pypdf import PdfReader

documents = []

for filename in uploaded_files:
    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text.strip(),
                "source": filename,
                "page": page_number
            })

print("✅ Text extraction complete")
print("Pages extracted:", len(documents))

for doc in documents[:2]:
    print("\nSOURCE:", doc["source"])
    print("PAGE:", doc["page"])
    print("TEXT PREVIEW:", doc["text"][:500])

✅ Text extraction complete
Pages extracted: 6

SOURCE: 3029.pdf
PAGE: 1
TEXT PREVIEW: International Journal on Science and Technology (IJSAT) 
E-ISSN: 2229-7677   ●   Website: www.ijsat.org   ●   Email: editor@ijsat.org 
 
IJSAT25013029 Volume 16, Issue 1, January-March 2025 1 
 
AI-Powered Fake Product Detection System 
 
Kanak Verma1, Priyanshi Bilgaiya2, Rishika Bhatia3, 
 Shriyanshi Bilgaiya4, Prof. Sonali Rathore5 
1, 2, 3, 4B.Tech Scholar, 5Assistant Professor 
Department of Artificial Intelligence & Data Science, 
Shri Balaji Institute of Technology & Management 
Betul, RG

SOURCE: 3029.pdf
PAGE: 2
TEXT PREVIEW: International Journal on Science and Technology (IJSAT) 
E-ISSN: 2229-7677   ●   Website: www.ijsat.org   ●   Email: editor@ijsat.org 
 
IJSAT25013029 Volume 16, Issue 1, January-March 2025 2 
 
holograms, barcodes, and manual inspections, are no longer sufficient due to the increasing 
sophistication of counterfeiters. 
Artificial Intelligence (AI) has emerged as a powe

# 4. Chunking

### Why chunk?

A large PDF may contain thousands of words. We divide it into smaller searchable pieces called **chunks**.

For this classroom lab:

- Chunk size = 500 words
- Overlap = 50 words

Each chunk keeps its `source` and `page` metadata.

In [20]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunks = []
chunk_id = 0

for document in documents:
    words = document["text"].split()
    start = 0

    while start < len(words):
        end = start + CHUNK_SIZE
        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_text,
                "source": document["source"],
                "page": document["page"]
            })
            chunk_id += 1

        start += CHUNK_SIZE - CHUNK_OVERLAP

print("✅ Chunking complete")
print("Total chunks:", len(chunks))

✅ Chunking complete
Total chunks: 7


In [21]:
for chunk in chunks[:3]:
    print("=" * 60)
    print("CHUNK:", chunk["chunk_id"])
    print("SOURCE:", chunk["source"])
    print("PAGE:", chunk["page"])
    print("TEXT:", chunk["text"][:400])

CHUNK: 0
SOURCE: 3029.pdf
PAGE: 1
TEXT: International Journal on Science and Technology (IJSAT) E-ISSN: 2229-7677 ● Website: www.ijsat.org ● Email: editor@ijsat.org IJSAT25013029 Volume 16, Issue 1, January-March 2025 1 AI-Powered Fake Product Detection System Kanak Verma1, Priyanshi Bilgaiya2, Rishika Bhatia3, Shriyanshi Bilgaiya4, Prof. Sonali Rathore5 1, 2, 3, 4B.Tech Scholar, 5Assistant Professor Department of Artificial Intelligenc
CHUNK: 1
SOURCE: 3029.pdf
PAGE: 2
TEXT: International Journal on Science and Technology (IJSAT) E-ISSN: 2229-7677 ● Website: www.ijsat.org ● Email: editor@ijsat.org IJSAT25013029 Volume 16, Issue 1, January-March 2025 2 holograms, barcodes, and manual inspections, are no longer sufficient due to the increasing sophistication of counterfeiters. Artificial Intelligence (AI) has emerged as a powerful tool in combating counterfeit products.
CHUNK: 2
SOURCE: 3029.pdf
PAGE: 2
TEXT: and gradient descent optimization techniques to minimize classification errors

# 5. Embeddings

An **embedding** converts text into a numerical vector representing its meaning.

We use the free local model:

`all-MiniLM-L6-v2`

So every chunk can be compared with an student question without an API call.

In [22]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("✅ Embeddings created")
print("Number of vectors:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings created
Number of vectors: 7
Vector dimensions: 384


# 6. Vector Database

We need to store:

- chunks
- embeddings
- metadata

We use **ChromaDB**, a local vector database.

Production systems can use PostgreSQL + pgvector or another vector database.

In [24]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="study_materials"
)

collection.add(
    ids=[str(chunk["chunk_id"]) for chunk in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "source": chunk["source"],
            "page": chunk["page"]
        }
        for chunk in chunks
    ]
)

print("✅ Vector database ready")
print("Stored chunks:", collection.count())

✅ Vector database ready
Stored chunks: 7


# 7. Ask a Natural-Language Question

Now we move to the query side.

Example:

> How many days of annual leave can I take?

The question will also be converted into an embedding.

In [26]:
question = input("Ask an student question: ")

query_embedding = embedding_model.encode(
    [question]
)[0].tolist()

print("\nQUESTION:", question)
print("✅ Query embedding created")

Ask an student question: algorithm

QUESTION: algorithm
✅ Query embedding created


# 8. Similarity Search + Semantic Search

**Similarity Search:** find vectors closest to the question vector.

**Semantic Search:** because embeddings represent meaning, relevant text can be found even when the exact words differ.

We retrieve the Top-K most relevant chunks.

In [27]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=min(3, len(chunks))
)

print("✅ Similarity search complete")

✅ Similarity search complete


In [28]:
retrieved_chunks = []

for i, text in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]

    item = {
        "text": text,
        "source": metadata["source"],
        "page": metadata["page"],
        "distance": distance
    }

    retrieved_chunks.append(item)

    print("=" * 60)
    print("RANK:", i + 1)
    print("DISTANCE:", round(distance, 4))
    print("SOURCE:", metadata["source"])
    print("PAGE:", metadata["page"])
    print("TEXT:", text[:700])

RANK: 1
DISTANCE: 1.3672
SOURCE: 3029.pdf
PAGE: 2
TEXT: and gradient descent optimization techniques to minimize classification errors and enhance accuracy.
RANK: 2
DISTANCE: 1.5007
SOURCE: 3029.pdf
PAGE: 5
TEXT: International Journal on Science and Technology (IJSAT) E-ISSN: 2229-7677 ● Website: www.ijsat.org ● Email: editor@ijsat.org IJSAT25013029 Volume 16, Issue 1, January-March 2025 5 5. Conclusion In server side there are two components a web server and Machine Learning Approach. When user sends an image through website the image will be taken by server and verified by machine learning model. In addition, the server also performs several operations such as storing detection results, data statistic or allowing users to report counterfeit products. The machine learning application is the main contribution of this paper. This solution provides a low-cost implementation, which is appropriate when t
RANK: 3
DISTANCE: 1.5538
SOURCE: 3029.pdf
PAGE: 3
TEXT: International Journal on Scien

# 9. Retrieval

This is the **Retrieval** part of RAG.

```text
student Question
       ↓
Query Embedding
       ↓
Similarity Search
       ↓
Top-K Relevant Chunks
```

The system has found evidence before generating an answer.

# 10. Augmentation

Now we put the retrieved evidence into a prompt.

This is the **Augmentation** part of RAG.

The language model will receive:

1. The student's question
2. The retrieved study material

In [30]:
context = "\n\n".join(
    f"SOURCE: {chunk['source']}\n"
    f"PAGE: {chunk['page']}\n"
    f"CONTENT: {chunk['text']}"
    for chunk in retrieved_chunks
)

prompt = (
    "You are an ExamBuddy, an AI Study Assistant.\n\n"
    "Answer the question using ONLY the provided study material context. "
    "Do not invent facts or answers.\n\n"
    "QUESTION:\n" + question + "\n\n"
    "STUDY MATERIAL CONTEXT:\n" + context
)

print("✅ RAG prompt created")
print(prompt[:3500])

✅ RAG prompt created
You are an ExamBuddy, an AI Study Assistant.

Answer the question using ONLY the provided study material context. Do not invent facts or answers.

QUESTION:
algorithm

STUDY MATERIAL CONTEXT:
SOURCE: 3029.pdf
PAGE: 2
CONTENT: and gradient descent optimization techniques to minimize classification errors and enhance accuracy.

SOURCE: 3029.pdf
PAGE: 5
CONTENT: International Journal on Science and Technology (IJSAT) E-ISSN: 2229-7677 ● Website: www.ijsat.org ● Email: editor@ijsat.org IJSAT25013029 Volume 16, Issue 1, January-March 2025 5 5. Conclusion In server side there are two components a web server and Machine Learning Approach. When user sends an image through website the image will be taken by server and verified by machine learning model. In addition, the server also performs several operations such as storing detection results, data statistic or allowing users to report counterfeit products. The machine learning application is the main contribution of this p

# 11. Generation — Local LLM

Now we perform **Generation**.

We use a small instruction-following model locally in Colab.

If available, use a GPU:

**Runtime → Change runtime type → T4 GPU**

This keeps the practical free of API charges.

In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading local LLM on:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("✅ Local LLM loaded")

Loading local LLM on: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Local LLM loaded


In [32]:
messages = [
    {
        "role": "system",
        "content": "You are ExamBuddy, an AI assistant. Use only the supplied company context. Do not invent policies."
    },
    {
        "role": "user",
        "content": prompt
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=4096
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("ANSWER")
print("=" * 60)
print(answer)

ANSWER
Algorithm:

1. **Data Collection**: Collect relevant data about images, including their dimensions, types, and content.
2. **Feature Extraction**: Convert the images into numerical feature vectors using techniques like Principal Component Analysis (PCA).
3. **Model Training**: Train a Support Vector Machine (SVM) classifier using the extracted feature vectors.
4. **Prediction**: Use the trained SVM model to predict whether an image belongs to the 'fake' or 'real' category.
5. **Classification Accuracy**: Evaluate the model's performance by comparing predicted labels against actual labels.
6. **Optimization Techniques**: Implement Gradient Descent Optimization to adjust parameters during training.
7. **Deployment**: Deploy the optimized model for real-time fraud detection in websites.
8. **Monitoring**: Continuously monitor system performance and update models as needed.
9. **Scalability**: Ensure the model scales well with increasing numbers of users and data volumes.
10. **Secu

# 12. Citations

Because we kept `source` and `page` as metadata, we can show where the retrieved evidence came from.

This gives the user a **source-backed answer**.

In [33]:
print("SOURCES USED")
print("-" * 50)

seen = set()

for chunk in retrieved_chunks:
    citation = (chunk["source"], chunk["page"])

    if citation not in seen:
        print(f"• {chunk['source']} — Page {chunk['page']}")
        seen.add(citation)

SOURCES USED
--------------------------------------------------
• 3029.pdf — Page 2
• 3029.pdf — Page 5
• 3029.pdf — Page 3


# 13. Multi-Document Search

All uploaded PDFs are stored in the same vector database.

Therefore one student question can search across:

```text
mobile application development uint 1
data structure and algorithms notes
or course syllabus
student Handbook
Travel Policy
        ↓
   One Vector DB
        ↓
Relevant chunks
```

Try questions about different policies and observe the source documents returned.

In [34]:
print("Documents in the knowledge base:")

for source in sorted(set(chunk["source"] for chunk in chunks)):
    print(" -", source)

print("\nTotal searchable chunks:", collection.count())

Documents in the knowledge base:
 - 3029.pdf

Total searchable chunks: 7


# 14. Evaluation

A RAG system must be evaluated.

### Retrieval
Did we retrieve the correct chunks?

### Answer
Did the answer correctly use those chunks?

### Groundedness
Are the answer's claims supported by the retrieved documents?

### Citation correctness
Does the cited document/page actually support the answer?

In [35]:
print("Retrieved evidence for evaluation:")
for rank, chunk in enumerate(retrieved_chunks, start=1):
    print(
        f"{rank}. {chunk['source']} | "
        f"Page {chunk['page']} | "
        f"distance={chunk['distance']:.4f}"
    )

Retrieved evidence for evaluation:
1. 3029.pdf | Page 2 | distance=1.3672
2. 3029.pdf | Page 5 | distance=1.5007
3. 3029.pdf | Page 3 | distance=1.5538


# 15. Final RAG Pipeline

```text
PDFs
 ↓
Text Extraction
 ↓
Chunking
 ↓
Metadata
 ↓
Embeddings
 ↓
Vector Database
 ↓
User Question
 ↓
Query Embedding
 ↓
Similarity / Semantic Search
 ↓
Relevant Chunks
 ↓
Augmented Prompt
 ↓
LLM Generation
 ↓
Source-backed Answer
 ↓
Citations
```

## Syllabus coverage

- ✅ Embeddings
- ✅ Vector Databases
- ✅ Semantic Search
- ✅ Chunking Strategies
- ✅ Metadata
- ✅ Similarity Search
- ✅ RAG Pipelines
- ✅ Evaluation

## Hands-on features

- ✅ Upload PDFs
- ✅ Ask natural-language questions
- ✅ Source-backed answers
- ✅ View citations
- ✅ Multi-document search

## Industry Challenge

**HR policy assistant capable of answering employee queries using company documents.**